# Jobs sourcing 

In [ ]:
from autocandid.greenhouse_ingestor import GreenhouseIngestor

In [3]:

# Let's target Discord as our safe PoC target
ingestor = GreenhouseIngestor(company_board_token="discord")

try:
    jobs = await ingestor.fetch_active_jobs()
    

    # Print the first job to verify our data contract is perfect
    sample_job = jobs[0]
    print("\n" + "="*50)
    print("SAMPLE EXTRACTED JOB DATA (READY FOR LLM MATCHING)")
    print("="*50)
    print(f"ID:       {sample_job.external_id}")
    print(f"Title:    {sample_job.title}")
    print(f"Company:  {sample_job.company}")
    print(f"Location: {sample_job.location}")
    print(f"URL:      {sample_job.url}")
    print(f"Desc Snippet: {sample_job.description[:150]}...\n")
    
except httpx.HTTPError as e:
    print(f"[!] HTTP Network Error: {e}")

[*] Fetching live jobs for 'discord' from Greenhouse API...
[+] Successfully validated and ingested 49/49 jobs.

SAMPLE EXTRACTED JOB DATA (READY FOR LLM MATCHING)
ID:       8806482002
Title:    Commercial Policy Lead
Company:  DISCORD
Location: San Francisco Bay Area
URL:      https://job-boards.greenhouse.io/discord/jobs/8806482002
Desc Snippet: <div class="content-intro"><p>Discord has a highly engaged community of millions of daily active users who use the platform for many different reasons...



In [ ]:
company_token = 'discord'
api_url = f"https://boards-api.greenhouse.io/v1/boards/{company_token}/jobs?content=true"

api_url

In [ ]:
validated_jobs

In [ ]:
jobs_data

# Job filtering: Stage 1 deterministic heuristic gate

In [4]:
import re
from typing import List, Tuple, Optional
from pydantic import BaseModel

In [6]:
class JobPosting(BaseModel):
    id: str
    title: str
    location: str
    requires_visa: bool
    is_remote: bool

In [9]:
# ============================================================================
# 2. RULE INTERFACES
# ============================================================================
class FilterRule:
    """Base class for all deterministic rules."""
    def evaluate(self, job: JobPosting) -> Tuple[bool, Optional[str]]:
        raise NotImplementedError

class TitleTaxonomyRule(FilterRule):
    def __init__(self):
        # Using word boundaries (\b) to prevent "Data Entry" from passing just because of "Data"
        self.pattern = re.compile(
            r'\b(data scientist|data science|machine learning|ml|ai|artificial intelligence|deep learning|llm|quant)\b', 
            re.IGNORECASE
        )
        
    def evaluate(self, job: JobPosting) -> Tuple[bool, Optional[str]]:
        # Normalize text by lowercasing before regex matching
        normalized_title = job.title.lower().strip()
        
        if self.pattern.search(normalized_title):
            return True, None
        return False, f"Title '{job.title}' does not match target AI/Data taxonomy."

class VisaRule(FilterRule):
    def __init__(self, user_needs_visa: bool):
        self.user_needs_visa = user_needs_visa

    def evaluate(self, job: JobPosting) -> Tuple[bool, Optional[str]]:
        if self.user_needs_visa and not job.requires_visa:
            return False, "Job does not offer required Visa sponsorship."
        if not self.user_needs_visa and job.requires_visa:
            # If the job strictly requires a visa applicant and we don't need one, 
            # or if it's a proxy for 'only hiring foreign contractors' (rare but happens).
            # Usually, you just want to avoid jobs that *cannot* sponsor if you need it.
            pass 
        
        return True, None

class ModalityRule(FilterRule):
    def __init__(self, requires_remote: bool):
        self.requires_remote = requires_remote

    def evaluate(self, job: JobPosting) -> Tuple[bool, Optional[str]]:
        if self.requires_remote and not job.is_remote:
            return False, f"Job in '{job.location}' is not remote."
        return True, None

# ============================================================================
# 3. THE HEURISTIC GATE (ORCHESTRATOR)
# ============================================================================
class HeuristicGate:
    def __init__(self, rules: List[FilterRule]):
        self.rules = rules

    def process(self, job: JobPosting) -> Tuple[bool, List[str]]:
        """Returns True if the job passes ALL rules, plus the rejection reason if any."""
        for rule in self.rules:
            passed, reason = rule.evaluate(job)
            if not passed:
                # Short-circuit: fail immediately upon first broken rule
                return False, [reason]
        return True, []

In [7]:
# ---------------------------------------------------------
# Test Data: Let's see how the engine handles different edge cases
# ---------------------------------------------------------
test_jobs = [
    JobPosting(
        id="job_1", title="Senior Machine Learning Engineer", 
        location="San Francisco, CA", requires_visa=False, is_remote=True
    ),
    JobPosting(
        id="job_2", title="Frontend Developer - AI Team", 
        location="New York, NY", requires_visa=False, is_remote=True
    ),
    JobPosting(
        id="job_3", title="Data Scientist", 
        location="Austin, TX", requires_visa=False, is_remote=False
    ),
    JobPosting(
        id="job_4", title="Lead Data Entry Specialist", 
        location="Remote", requires_visa=False, is_remote=True
    )
]

test_jobs

[JobPosting(id='job_1', title='Senior Machine Learning Engineer', location='San Francisco, CA', requires_visa=False, is_remote=True),
 JobPosting(id='job_2', title='Frontend Developer - AI Team', location='New York, NY', requires_visa=False, is_remote=True),
 JobPosting(id='job_3', title='Data Scientist', location='Austin, TX', requires_visa=False, is_remote=False),
 JobPosting(id='job_4', title='Lead Data Entry Specialist', location='Remote', requires_visa=False, is_remote=True)]

In [12]:
# Configure your personal constraints here
my_rules = [
    TitleTaxonomyRule(),
    VisaRule(user_needs_visa=False), # Assuming you have US work auth
    ModalityRule(requires_remote=False) # Let's say you only want remote roles right now
]

gate = HeuristicGate(rules=my_rules)

for job in test_jobs:
    print(f"Evaluating: {job.title} ({job.location} | Remote: {job.is_remote})")
    passed, reasons = gate.process(job)
    
    if passed:
        print("  -> [✓] PASSED! Route to LLM.\n")
    else:
        print(f"  -> [X] REJECTED: {reasons[0]}\n")

Evaluating: Senior Machine Learning Engineer (San Francisco, CA | Remote: True)
  -> [✓] PASSED! Route to LLM.

Evaluating: Frontend Developer - AI Team (New York, NY | Remote: True)
  -> [✓] PASSED! Route to LLM.

Evaluating: Data Scientist (Austin, TX | Remote: False)
  -> [✓] PASSED! Route to LLM.

Evaluating: Lead Data Entry Specialist (Remote | Remote: True)
  -> [X] REJECTED: Title 'Lead Data Entry Specialist' does not match target AI/Data taxonomy.



# Workflow Testing 

In [13]:
import asyncio
from autocandid.greenhouse_ingestor import GreenhouseIngestor
from autocandid.heuristic_gate import (
    HeuristicGate, 
    TitleTaxonomyRule, 
    LocationRule,
    AntiKeywordRule
)

In [20]:
print("="*50)
print("AUTOCANDID PIPELINE: STAGE 1 (INGEST & GATE)")
print("="*50)

# 1. Initialize dependencies
# Testing with 'discord' as a reliable Greenhouse endpoint
ingestor = GreenhouseIngestor(company_board_token="discord") 

# Configure strict heuristic rules
gate = HeuristicGate(rules=[
    AntiKeywordRule(),       # Quickest string match to drop obvious trash
    TitleTaxonomyRule(),     # Core target matching
    LocationRule(require_remote=False) # Set to True if you only want remote roles
])

# 2. Extract Data
raw_jobs = await ingestor.fetch_active_jobs()
if not raw_jobs:
    print("[-] No jobs returned from ingestor. Exiting.")
    # return

# 3. Filter Data
surviving_jobs = []
print("\n[*] Applying Heuristic Gate to ingested jobs...")


for job in raw_jobs:
    passed, reasons = gate.process(job)
    if passed:
        surviving_jobs.append(job)
        # Silencing the successes to keep terminal clean, just logging failures below
    else:
        print(f"  [X] Pruned [{job.external_id}]: {job.title} -> {reasons[0]}")


print("\n" + "="*50)
print(f"PIPELINE SUMMARY")
print("="*50)
print(f"Total Ingested:  {len(raw_jobs)}")
print(f"Total Surviving: {len(surviving_jobs)}")
print(f"Prune Rate:      {((len(raw_jobs) - len(surviving_jobs)) / len(raw_jobs)) * 100:.1f}%\n")
    
if surviving_jobs:
    print("SURVIVOR SAMPLE (Ready for LLM):")
    sample = surviving_jobs[0]
    print(f" - {sample.title} ({sample.location}) | ID: {sample.external_id}")


AUTOCANDID PIPELINE: STAGE 1 (INGEST & GATE)
[*] Fetching live jobs for 'discord' from Greenhouse API...
[+] Successfully validated and ingested 49/49 jobs.

[*] Applying Heuristic Gate to ingested jobs...
  [X] Pruned [8806482002]: Commercial Policy Lead -> Title 'Commercial Policy Lead' misses AI/Data taxonomy.
  [X] Pruned [8571766002]: Director of Engineering, Safety -> Title 'Director of Engineering, Safety' misses AI/Data taxonomy.
  [X] Pruned [8537955002]: Engineering Manager, Notifications -> Title 'Engineering Manager, Notifications' misses AI/Data taxonomy.
  [X] Pruned [8649856002]: Engineering Manager, Platform Security -> Title 'Engineering Manager, Platform Security' misses AI/Data taxonomy.
  [X] Pruned [8637688002]: Engineering Manager, Safety -> Title 'Engineering Manager, Safety' misses AI/Data taxonomy.
  [X] Pruned [8827890002]: Graphic Designer & Illustrator -> Title 'Graphic Designer & Illustrator' misses AI/Data taxonomy.
  [X] Pruned [8801465002]: Marketing Pro